## Inserción de datos

En el siguiente notebook se hace la inserción de datos correspondiente para las tablas creadas en el notebook 01_setup_schema usando DataFrames de Spark y comandos de Python y PySpark. 

Las celdas correspondientes al injesto de datos siguen un mismo patron: primero se crea una lista vacia, despues se llenan las columnas usando las librerias random y faker en dicha lista usando for e in rango la cantidad de datos que se solicito, posteriormente se crea un dataframe y por ultimo se inserta en las tablas que se crearon en el notebook setup schema.

1. Importación de librerias y configuramos la cantidad de datos deseadas para las tablas:

In [0]:
# Instalación
%pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Importaciones y configuración
import random
from faker import Faker
from pyspark.sql import Row
from datetime import datetime, timedelta

fake = Faker("es_ES")

CANTIDAD_USUARIOS = 800
CANTIDAD_EVENTOS = 60
DB = "workspace.apuestas_db"
HOY = datetime.now()

# Probabilidad de que un usuario nazca "en riesgo"
PROB_EN_RIESGO = 0.4

2. Datos para la tabla estado_apuesta

In [0]:
estados = [
    Row(id_estado=1, nombre_estado="Pendiente"),
    Row(id_estado=2, nombre_estado="Ganada"),
    Row(id_estado=3, nombre_estado="Perdida"),
]
df_estados = spark.createDataFrame(estados)
df_estados.createOrReplaceTempView("tmp_estados")
spark.sql(f"""
    INSERT INTO {DB}.estado_apuesta (id_estado, nombre_estado)
    SELECT id_estado, nombre_estado
    FROM tmp_estados
""")
print("✔ estado_apuesta poblada")

✔ estado_apuesta poblada


3. Datos para la tabla usuarios mas la creación de perfiles ocultos (Para esto se crean dos tipos de perfiles, generando datos según estos, el primero será un perfil activo y el segundo será un perfil en riesgo, la idea es que el modelo pueda predecir a partir de comportamientos reales que asemejen el riesgo de un cliente al abandonar la plataforma)

In [0]:
# 3. Usuarios + perfil oculto (los perfiles NO se inserta en la tabla, solo se clasifican para generar los datos de manera aleatoria)

usuarios = []
perfiles = {}  # diccionario auxiliar: se usará luego para generar apuestas/transacciones

for i in range(CANTIDAD_USUARIOS):
    fecha_registro = fake.date_time_between(start_date="-1y", end_date="-2M")
    perfil = "en_riesgo" if random.random() < PROB_EN_RIESGO else "activo"

    usuarios.append(Row(
        nombre=fake.first_name(),
        apellido=fake.last_name(),
        numero_documento=str(fake.random_number(digits=8, fix_len=True)),
        tipo_documento=random.choice(["DNI", "Pasaporte", "Cédula"]),
        fecha_registro=fecha_registro,
        saldo=float(round(random.uniform(1000, 50000), 2))
    ))
    perfiles[i] = perfil  # guardamos el perfil por posición, aun no se asocia al id del usuario

#creamos un dataframe y lo insertamos en la tabla usuarios de nuestra BD
df_usuarios = spark.createDataFrame(usuarios)
df_usuarios.createOrReplaceTempView("tmp_usuarios")
spark.sql(f"""
    INSERT INTO {DB}.usuario (nombre, apellido, numero_documento, tipo_documento, fecha_registro, saldo)
    SELECT nombre, apellido, numero_documento, tipo_documento, fecha_registro, saldo
    FROM tmp_usuarios
""")

# Se hace una consulta en SQL para retornar todos los id generados, usamos collect para traer ese resultado a Python, finalmente el ciclo for guarda todos los ids en la lista ids_usuarios_ordenados
ids_usuarios_ordenados = [r.id_usuario for r in spark.sql(
    f"SELECT id_usuario FROM {DB}.usuario ORDER BY id_usuario" 
).collect()]

# Mapeamos id_usuario real con el perfil simulado, se crea un diccionario, se le asigna como llave los ids ordenados que se retornaron antes, y como argumento el tipo de perfil asignado
perfil_por_id = {ids_usuarios_ordenados[i]: perfiles[i] for i in range(CANTIDAD_USUARIOS)}

print("usuarios generados con perfil oculto asignado")

usuarios generados con perfil oculto asignado


4. Datos para la tabla eventos, para esta tabla hacemos el supuesto de que la plataforma realiza apuestas para 5 tipos de deporte distintos:

In [0]:
# 4. Eventos 
deportes = ["Fútbol", "Baloncesto", "Tenis", "Béisbol", "Ciclismo"]
equipos = {
    "Fútbol":     ["Real Madrid", "Barcelona", "Millonarios", "Nacional", "Bayern", "PSG"],
    "Baloncesto": ["Lakers", "Bulls", "Heat", "Celtics", "Warriors"],
    "Tenis":      ["Alcaraz", "Djokovic", "Sinner", "Medvedev", "Zverev"],
    "Béisbol":    ["Yankees", "Dodgers", "Red Sox", "Cubs", "Astros"],
    "Ciclismo":   ["Bernal", "Pogacar", "Vingegaard", "Roglic", "Thomas"],
}

eventos = []
for _ in range(CANTIDAD_EVENTOS):
    deporte = random.choice(deportes)
    participantes_lista = random.sample(equipos[deporte], 2)
    fecha_ev = fake.date_time_between(start_date="-2M", end_date="+1M")
    resultado = random.choice(["Pendiente", participantes_lista[0], participantes_lista[1], "Empate"])
    eventos.append(Row(
        tipo_deporte=deporte,
        participantes=" vs ".join(participantes_lista),
        fecha_evento=fecha_ev,
        resultado=resultado
    ))

df_eventos = spark.createDataFrame(eventos)
df_eventos.createOrReplaceTempView("tmp_eventos")
spark.sql(f"""
    INSERT INTO {DB}.evento (tipo_deporte, participantes, fecha_evento, resultado)
    SELECT tipo_deporte, participantes, fecha_evento, resultado
    FROM tmp_eventos
""")

# Recuperamos todos los id_evento generados automáticamente en la tabla 'evento' y los almacena en una lista de Python. 
# Estos IDs se utilizarán posteriormente para asociar las apuestas a eventos existentes, manteniendo la integridad referencial entre ambas tablas.

ids_eventos = [r.id_evento for r in spark.sql(f"SELECT id_evento FROM {DB}.evento").collect()]
print("eventos generados")


eventos generados


5. Datos para la tabla apuestas, en este codigo realizamos el sesgo real de datos entre los dos perfiles de usuarios que creamos de manera artificial:

- Usuarios activos: realizan muchas apuestas, han apostado recientemente, los montos son relativamente altos y tienen una tasa de pérdida cercana al 50%.
- Usuarios en riesgo: realizan pocas apuestas, dejaron de apostar hace semanas o meses, los montos disminuyen con el tiempo y presentan una mayor probabilidad de perder.


In [0]:
# 5. Apuestas
apuestas = []

for id_usr in ids_usuarios_ordenados:
    # recuperamos el perfil oculto que habiamos generado para cada id
    perfil = perfil_por_id[id_usr]

    if perfil == "activo":
        # Los usuarios activos realizan entre 8 y 15 apuestas, mantienen actividad reciente (últimos 15 días) y presentan una tasa de pérdida cercana al 50%.
        n_apuestas = random.randint(20, 35)
        ventana_dias_max = 180
        tasa_perdida = random.uniform(0.45, 0.55)
    else:  
        # Los usuarios en riesgo realizan pocas apuestas, llevan mucho tiempo sin actividad y pierden con mayor frecuencia.
        n_apuestas = random.randint(1, 5)
        ventana_dias_max = random.randint(45, 180)
        tasa_perdida = random.uniform(0.65, 0.80)

    
    # Generamos la cantidad de apuestas definida para cada perfil de usuario.

    for j in range(n_apuestas):
        if perfil == "activo":
            dias_atras = random.randint(0, ventana_dias_max)
        else:
            # concentramos las apuestas en la parte antigua de la ventana (dejaron de apostar)
            dias_atras = random.randint(ventana_dias_max - 20, ventana_dias_max)

        # Tendencia de monto: decreciente para en_riesgo (apuestan cada vez menos)
        if perfil == "activo":
            monto = round(random.uniform(200, 5000), 2)
        else:
            factor_decrecimiento = 1 - (j / max(n_apuestas, 1)) * 0.5
            monto = round(random.uniform(100, 2000) * factor_decrecimiento, 2)


        # Con una probabilidad MENOR a la tasa de pérdida la apuesta se marca como Perdida (3),en otro caso se asigna aleatoriamente como pendiente o ganada
        estado = 3 if random.random() < tasa_perdida else random.choice([1, 2]) 
        
        # Construimos el registro de la apuesta:
        apuestas.append(Row(
            id_usuario=id_usr,
            id_evento=random.choice(ids_eventos),
            monto=float(monto),
            cuota=float(round(random.uniform(1.1, 10.0), 2)),
            fecha_apuesta=HOY - timedelta(days=dias_atras),
            id_estado_apuesta=estado
        ))

# Se convierte la lista en un DF y se inserta en la tabla correspondiente
df_apuestas = spark.createDataFrame(apuestas)
df_apuestas.createOrReplaceTempView("tmp_apuestas")
spark.sql(f"""
    INSERT INTO {DB}.apuesta (id_usuario, id_evento, monto, cuota, fecha_apuesta, id_estado_apuesta)
    SELECT id_usuario, id_evento, monto, cuota, fecha_apuesta, id_estado_apuesta
    FROM tmp_apuestas
""")
print("apuestas generadas con sesgo de perfil")


apuestas generadas con sesgo de perfil


6. Datos para la tabla transacciones:

In [0]:
# 6. Transacciones

# Traemos las apuestas ya generadas y agrupadas por usuario con su fecha
apuestas_por_usuario = {}
rows_apuestas = spark.sql(f"""
    SELECT id_usuario, id_apuesta, fecha_apuesta, id_estado_apuesta, monto
    FROM {DB}.apuesta
""").collect()

# agrupamos las apuestas con un diccionario, donde la llave es el id_usuario y el valor es una lista de apuestas
for r in rows_apuestas:
    apuestas_por_usuario.setdefault(r.id_usuario, []).append(r)

transacciones = []

# seleccionamos las transacciones para cada usuario segun su perfil asignado antes
for id_usr in ids_usuarios_ordenados:
    perfil = perfil_por_id[id_usr]
    apuestas_usr = apuestas_por_usuario.get(id_usr, [])

    # 1.Transacciones: por cada apuesta ya resuelta (ganada o perdida) se genera una transacción de tipo "pago_apuesta".
    for ap in apuestas_usr:
        if ap.id_estado_apuesta in (2, 3):  # 2=Ganada, 3=Perdida (no se paga una apuesta que esta pendiente)
            fecha_pago = ap.fecha_apuesta + timedelta(days=random.randint(0, 2))
            transacciones.append(Row(
                id_usuario=id_usr,
                id_apuesta=ap.id_apuesta,
                monto=float(ap.monto),
                tipo_transaccion="pago_apuesta",
                fecha_transaccion=fecha_pago
            ))

    # 2.Depósitos y retiros: no depende de una apuesta, su frecuencia y antigüedad dependen del perfil del usuario.
    if perfil == "activo":
        # Los usuarios activos realizan más movimientos financieros y mas recientes
        n_dep_ret = random.randint(8, 14)
        ventana_dias_max = 180  
    else:  
        #Los usuarios en riesgo realizan pocos movimientos y hace mucho tiempo no tienen actividad financiera.
        n_dep_ret = random.randint(1, 3)
        ventana_dias_max = random.randint(45, 180) 


    # Se generan los depósitos y retiros correspondientes:

    for _ in range(n_dep_ret):
        if perfil == "activo":
            # movimientos recientes
            dias_atras = random.randint(0, ventana_dias_max)
        else:
            # movimientos antiguos
            dias_atras = random.randint(ventana_dias_max - 20, ventana_dias_max)

        transacciones.append(Row(
            id_usuario=id_usr,
            id_apuesta=None,  # depósito/retiro no tienen una apuesta específica
            monto=float(round(random.uniform(100, 5000), 2)),
            # se selecciona aleatoriamente si la transacción es un depósito o un retiro
            tipo_transaccion=random.choice(["deposito", "retiro"]),
            fecha_transaccion=HOY - timedelta(days=dias_atras)
        ))

# se convierte la lista en DF y se inserta en la tabla correspondiente
df_transacciones = spark.createDataFrame(transacciones)
df_transacciones.createOrReplaceTempView("tmp_transacciones")
spark.sql(f"""
    INSERT INTO {DB}.transaccion (id_usuario, id_apuesta, monto, tipo_transaccion, fecha_transaccion)
    SELECT id_usuario, id_apuesta, monto, tipo_transaccion, fecha_transaccion
    FROM tmp_transacciones
""")
print("transacciones generadas")


transacciones generadas


VALIDACION FINAL: las siguientes celdas tienen como proposito verificar que los datos fueron llenados de manera correcta.

In [0]:
print("CONTEO DE REGISTROS")
for tabla in ["estado_apuesta", "usuario", "evento", "apuesta", "transaccion"]:
    count = spark.sql(f"SELECT COUNT(*) as total FROM {DB}.{tabla}").collect()[0].total
    print(f"  {tabla}: {count} registros")

CONTEO DE REGISTROS
  estado_apuesta: 3 registros
  usuario: 800 registros
  evento: 60 registros
  apuesta: 13957 registros
  transaccion: 16397 registros


Verifiquemos que no hay nulos en fecha_apuesta:

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA apuestas_db;

SELECT COUNT(*) as apuestas_sin_fecha
FROM apuesta
WHERE fecha_apuesta IS NULL;

apuestas_sin_fecha
0


Verificando que no hay registros huerfanos:

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA apuestas_db;
-- Apuestas con usuario o evento inexistente
SELECT COUNT(*) as apuestas_huerfanas
FROM apuesta a
LEFT JOIN usuario u ON a.id_usuario = u.id_usuario
LEFT JOIN evento e ON a.id_evento = e.id_evento
WHERE u.id_usuario IS NULL OR e.id_evento IS NULL;

-- Transacciones con usuario inexistente, o apuesta inexistente (cuando no es NULL)
SELECT COUNT(*) as transacciones_huerfanas
FROM transaccion t
LEFT JOIN usuario u ON t.id_usuario = u.id_usuario
WHERE u.id_usuario IS NULL;

transacciones_huerfanas
0


Verificando el sesgo del perfil

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA apuestas_db;

SELECT
    u.id_usuario,
    MAX(a.fecha_apuesta) as ultima_apuesta,
    DATEDIFF(CURRENT_TIMESTAMP(), MAX(a.fecha_apuesta)) as dias_desde_ultima_apuesta,
    COUNT(a.id_apuesta) as n_apuestas
FROM usuario u
LEFT JOIN apuesta a ON u.id_usuario = a.id_usuario
GROUP BY u.id_usuario
ORDER BY dias_desde_ultima_apuesta DESC

id_usuario,ultima_apuesta,dias_desde_ultima_apuesta,n_apuestas
1722,2026-01-25T22:22:43.060Z,173,4
1842,2026-01-25T22:22:43.060Z,173,1
2301,2026-01-29T22:22:43.060Z,169,4
2353,2026-01-30T22:22:43.060Z,168,5
1619,2026-01-31T22:22:43.060Z,167,4
1706,2026-01-31T22:22:43.060Z,167,1
1813,2026-02-02T22:22:43.060Z,165,1
2354,2026-02-02T22:22:43.060Z,165,1
2018,2026-02-03T22:22:43.060Z,164,4
1811,2026-02-03T22:22:43.060Z,164,5


Proporcion aproximada de perfiles (40/60)

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA apuestas_db;

SELECT
    CASE WHEN dias_desde_ultima_apuesta > 40 THEN 'probable_en_riesgo' ELSE 'probable_activo' END as grupo,
    COUNT(*) as n_usuarios
FROM (
    SELECT u.id_usuario, DATEDIFF(CURRENT_TIMESTAMP(), MAX(a.fecha_apuesta)) as dias_desde_ultima_apuesta
    FROM usuario u
    LEFT JOIN apuesta a ON u.id_usuario = a.id_usuario
    GROUP BY u.id_usuario
)
GROUP BY grupo

grupo,n_usuarios
probable_activo,491
probable_en_riesgo,309


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA apuestas_db;

SELECT
    MIN(monto) as monto_min, MAX(monto) as monto_max, AVG(monto) as monto_promedio,
    MIN(cuota) as cuota_min, MAX(cuota) as cuota_max
FROM apuesta;

monto_min,monto_max,monto_promedio,cuota_min,cuota_max
79.51,4999.97,2471.846381,1.10,10.00


Las conclusiones de las validaciones son las siguientes:

 La distribución de perfiles obtenida fue de 491 usuarios activos (62%) y 309 usuarios en riesgo (38%), una proporción muy cercana a la configuración inicial del generador (60% activos y 40% en riesgo). En cuanto a las apuestas, las cuotas se generaron correctamente dentro del rango definido de 1.10 a 10.00. Adicional, los montos oscilaron entre 63.52 y 4,999.85, con un promedio de 2,333.95. Aunque el monto mínimo es inferior al límite inicial de 100, este comportamiento responde a la lógica implementada para los usuarios en riesgo: el algoritmo aplica un factor de decrecimiento que reduce progresivamente el valor de las apuestas conforme el usuario se acerca al abandono de la plataforma. De esta manera, los datos sintéticos reflejan un comportamiento más realista y coherente con la hipótesis de negocio planteada. 
